# IMUSA V2 — Multi-Account Colab Worker 3 (Fold 4, Calibration & Ensemble)
This notebook runs **Fold 4**, gathers all 5 fold checkpoints, performs **Nelder-Mead Post-Hoc Threshold Calibration**, and evaluates the **Full 5-Fold Probability Ensemble**.

In [1]:
# 1. Environment & GPU Setup
!nvidia-smi
!pip install -q uv
import os
import sys

if not os.path.exists("imusa-multimodal-sentiment"):
    !git clone https://github.com/shubhojit-mitra-dev/imusa-multimodal-sentiment.git
%cd /content/imusa-multimodal-sentiment
!git pull origin main
!pip install -e libs/imusa
sys.path.insert(0, "/content/imusa-multimodal-sentiment/libs/imusa/src")

Sat Sep 12 18:17:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Google Drive Integration & Automatic data.zip Handling
import os
import shutil

from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)
gdrive_zip = "/content/drive/MyDrive/data.zip"
local_zip = "/content/imusa-multimodal-sentiment/data.zip"

if os.path.exists(gdrive_zip):
    print("Found data.zip in Google Drive. Copying locally...")
    shutil.copy(gdrive_zip, local_zip)
elif not os.path.exists(local_zip):
    print("data.zip not found in Google Drive (MyDrive/data.zip).")
    print("Please select and upload data.zip from your computer now:")
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(".zip"):
            shutil.move(fname, local_zip)
            break

# Copy to Google Drive for future runs
if os.path.exists(local_zip) and not os.path.exists(gdrive_zip):
    print("Saving data.zip to Google Drive (MyDrive/data.zip) for future runs...")
    try:
        shutil.copy(local_zip, gdrive_zip)
        print("Saved to Google Drive.")
    except Exception as e:
        print(f"Note: Could not copy to Drive: {e}")

# Unzip dataset
!unzip -q -o /content/imusa-multimodal-sentiment/data.zip -d /content/imusa-multimodal-sentiment/
print("Dataset extracted to data/.")

Mounted at /content/drive
Found data.zip in Google Drive. Copying locally...
Dataset extracted to data/.


In [3]:
# 3. Run Fold 4 Training
!python scripts/train_kfold.py --fold 4 --model-version v2 --num-folds 5 --epochs 10 --lp-epochs 3

2026-09-12 18:18:51,811 - INFO - --- Starting Stratified K-Fold Training: Fold 5/5 (Version: v2) ---
2026-09-12 18:18:51,811 - INFO - Starting dataset cleaning pipeline on /content/imusa-multimodal-sentiment/data/train/train_punjabi_dataset.csv
2026-09-12 18:18:51,924 - INFO - Saved cleaned dataset (2891 rows) to /content/imusa-multimodal-sentiment/data/processed/train_clean.csv
     IMUSA Dataset Cleaning Report      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Pipeline Stage               ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ Total Raw Rows Parsed        │  3002 │
│ Dropped (Missing Category)   │     0 │
│ Dropped (Invalid Category)   │     0 │
│ Dropped (Missing Image File) │     0 │
│ Dropped (Duplicates)         │   111 │
│ Final Clean Dataset Size     │  2891 │
└──────────────────────────────┴───────┘
2026-09-12 18:18:52,090 - INFO - HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-09-12 18:18:52,175 - INFO - HTTP Request: HEAD htt

In [4]:
import glob
import os
import shutil

# 4. Import Fold 0-3 Checkpoints and OOF output files from Account 1 & Account 2
for fname in ["fold_0_1_outputs.zip", "fold_2_3_outputs.zip"]:
    gdrive_file = f"/content/drive/MyDrive/{fname}"
    local_file = f"/content/imusa-multimodal-sentiment/{fname}"
    if os.path.exists(gdrive_file):
        print(f"Found {fname} in Google Drive. Copying...")
        shutil.copy(gdrive_file, local_file)
    elif not os.path.exists(local_file):
        print(f"Please upload {fname}:")
        uploaded = files.upload()
        for u_name in uploaded.keys():
            if u_name == fname or u_name.endswith(".zip"):
                shutil.move(u_name, local_file)
                break

!unzip -o -q fold_0_1_outputs.zip
!unzip -o -q fold_2_3_outputs.zip

# Normalize any files from outputs/v1 to outputs/v2
os.makedirs("outputs/v2/checkpoints", exist_ok=True)
for f in glob.glob("outputs/v1/checkpoints/best_model_fold_*.pt"):
    dst = os.path.join("outputs/v2/checkpoints", os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)
for f in glob.glob("outputs/v1/oof_*_fold_*.npy"):
    dst = os.path.join("outputs/v2", os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)

print("All 5 fold outputs ready.")

Found fold_0_1_outputs.zip in Google Drive. Copying...
Found fold_2_3_outputs.zip in Google Drive. Copying...
All 5 fold outputs ready.


In [5]:
# 5. Run Post-Hoc Threshold Calibration across all 5 OOF validation files
!python scripts/train_kfold.py --calibrate --model-version v2

2026-09-12 18:45:14,666 - INFO - --- Performing Post-Hoc Threshold Calibration Across Available Folds (v2) ---
2026-09-12 18:45:14,826 - INFO - Threshold calibration complete: Uncalibrated Macro F1=0.4548 -> Calibrated Macro F1=0.4630 (+0.83%)
2026-09-12 18:45:14,826 - INFO - Optimal threshold vector: [np.float64(1.0319889299571519), np.float64(0.8517324253916743), np.float64(1.0506044887006292), np.float64(1.1507181935012356)]
2026-09-12 18:45:14,827 - INFO - Saved calibrated thresholds to /content/imusa-multimodal-sentiment/outputs/v2/calibration/thresholds.json
2026-09-12 18:45:14,827 - INFO - Calibration successful: Macro F1 = 0.4630


In [6]:
# 6. Generate Calibrated Ensemble Test Set Predictions
!python scripts/predict.py --model-version v2 --use-ensemble --use-calibration

2026-09-12 18:45:21,897 [INFO] __main__: Reading test dataset from /content/imusa-multimodal-sentiment/data/test/Test.csv...
2026-09-12 18:45:21,910 [INFO] __main__: Loaded 500 test samples.
2026-09-12 18:45:25,500 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/google/muril-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-12 18:45:25,512 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/muril-base-cased/afd9f36c7923d54e97903922ff1b260d091d202f/config.json "HTTP/1.1 200 OK"
2026-09-12 18:45:25,605 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/google/muril-base-cased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-12 18:45:25,615 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/muril-base-cased/afd9f36c7923d54e97903922ff1b260d091d202f/tokenizer_config.json "HTTP/1.1 200 OK"
2026-09-12 18:45:25,698 [INFO] httpx: HTTP Request: GET https:/